In [1]:
import cv2
import dlib
import numpy as np
import random
from pathlib import Path

In [2]:
detector = dlib.get_frontal_face_detector()
predictor_path = r"C:\Users\Pulse_GL66\Desktop\Occlusions\Landmark_predictor_path\shape_predictor_68_face_landmarks.dat"
predictor = dlib.shape_predictor(predictor_path)

In [3]:
def load_image(path):
    try:
        with open(path, 'rb') as f:
            data = np.frombuffer(f.read(), dtype=np.uint8)
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        return img
    except:
        return cv2.imread(path)

In [4]:
def add_upper_occlusions(img, landmarks):
    h, w = img.shape[:2]
    eye_points = [(landmarks.part(i).x, landmarks.part(i).y) for i in range(36, 48)]
    eyes_min_x = min(p[0] for p in eye_points)
    eyes_max_x = max(p[0] for p in eye_points)
    eyes_min_y = min(p[1] for p in eye_points)
    eyes_max_y = max(p[1] for p in eye_points)
    eyes_width = eyes_max_x - eyes_min_x
    eyes_height = eyes_max_y - eyes_min_y
    occ_width = int(eyes_width * random.uniform(1.2, 2.0))
    occ_height = int(eyes_height * random.uniform(1.6, 4.0))
    center_x = (eyes_min_x + eyes_max_x) // 2
    center_y = (eyes_min_y + eyes_max_y) // 2
    x = center_x - occ_width // 2
    y = center_y - occ_height // 2
    x += random.randint(-int(occ_width*0.15), int(occ_width*0.15))
    y += random.randint(-int(occ_height*0.1), int(occ_height*0.1))
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(w, x + occ_width)
    y2 = min(h, y + occ_height)
    if x2 <= x1 or y2 <= y1:
        return img
    actual_width = x2 - x1
    actual_height = y2 - y1
    color = (random.randint(0, 50), random.randint(0, 50), random.randint(0, 50))
    alpha = random.uniform(0.6, 0.95)
    shape = random.choice(['rectangle', 'ellipse'])
    overlay = img.copy() 
    if shape == 'rectangle':
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
    else:
        center = (x1 + actual_width // 2, y1 + actual_height // 2)
        axes = (actual_width // 2, actual_height // 2)
        cv2.ellipse(overlay, center, axes, 0, 0, 360, color, -1)
    result = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    return result

In [5]:
def add_lower_occlusions(img, landmarks):
    h, w = img.shape[:2]
    mouth_points = [(landmarks.part(i).x, landmarks.part(i).y) for i in range(48, 68)]
    mouth_min_x = min(p[0] for p in mouth_points)
    mouth_max_x = max(p[0] for p in mouth_points)
    mouth_min_y = min(p[1] for p in mouth_points)
    mouth_max_y = max(p[1] for p in mouth_points)
    mouth_width = mouth_max_x - mouth_min_x
    mouth_height = mouth_max_y - mouth_min_y
    occ_width = int(mouth_width * random.uniform(1.2, 2.4))
    occ_height = int(mouth_height * random.uniform(1.2, 2.4))
    center_x = (mouth_min_x + mouth_max_x) // 2
    center_y = (mouth_min_y + mouth_max_y) // 2
    x = center_x - occ_width // 2
    y = center_y - occ_height // 2
    x += random.randint(-int(occ_width*0.15), int(occ_width*0.15))
    y += random.randint(-int(occ_height*0.1), int(occ_height*0.1))
    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(w, x + occ_width)
    y2 = min(h, y + occ_height)
    if x2 <= x1 or y2 <= y1:
        return img
    actual_width = x2 - x1
    actual_height = y2 - y1
    color_options = [
        (random.randint(150, 220), random.randint(150, 220), random.randint(150, 220)),
        (random.randint(100, 180), random.randint(150, 220), random.randint(200, 240)),
        (random.randint(180, 220), random.randint(180, 220), random.randint(200, 240)),
    ]
    color = random.choice(color_options)
    alpha = random.uniform(0.7, 0.95)
    shape = random.choice(['rectangle', 'ellipse'])
    overlay = img.copy()
    if shape == 'rectangle':
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
    else:
        center = (x1 + actual_width // 2, y1 + actual_height // 2)
        axes = (actual_width // 2, actual_height // 2)
        cv2.ellipse(overlay, center, axes, 0, 0, 360, color, -1)
    result = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    return result

In [6]:
def apply_occlusions(input_dir, output_dir, occlusion_ratio=0.7):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    images = [p for p in input_dir.rglob('*') if p.is_file()]
    stats = {'upper': 0, 'lower': 0, 'no_face': 0, 'original': 0}
    for i, img_path in enumerate(images):
        img = load_image(img_path)
        if img is None:
            stats['no_face'] += 1
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = detector(gray)
        if len(faces) == 0:
            stats['no_face'] += 1
            rel_path = img_path.relative_to(input_dir)
            output_path = output_dir / rel_path
            output_path.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(output_path, img)
            continue
        face = faces[0]
        landmarks = predictor(gray, face)
        if random.random() < occlusion_ratio:
            if random.random() < 0.5:
                result = add_upper_occlusions(img, landmarks)
                stats['upper'] += 1
            else:
                result = add_lower_occlusions(img, landmarks)
                stats['lower'] += 1
        else:
            result = img
            stats['original'] += 1
        rel_path = img_path.relative_to(input_dir)
        output_path = output_dir / rel_path
        output_path.parent.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(output_path, result)
    print(f"Очки/повязка: {stats['upper']}")
    print(f"Маска: {stats['lower']}")
    print(f"Оригинал: {stats['original']}")
    print(f"Лицо не найдено: {stats['no_face']}")
    print(f"Всего обработано: {len(images)}")

In [ ]:
apply_occlusions(input_dir=r"D:\НИР\RAF-DB\DATASET\test", output_dir=r"D:\Occlusions\occluded_test_dataset", occlusion_ratio=0.7)

Очки/повязка: 796
Маска: 851
Оригинал: 665
Лицо не найдено: 756
Всего обработано: 3068
